# 🛒 Рекомендательная система товаров в электронной коммерции

## 📌 Описание проекта

Электронная коммерция является одной из ключевых областей применения рекомендательных систем. Такие системы позволяют пользователям быстрее находить интересующие их товары, а бизнесу — увеличивать вовлечённость и выручку.

В рамках данного проекта решается задача:

> **предсказания товаров, которые с наибольшей вероятностью будут добавлены пользователем в корзину**

Рекомендации строятся на основе пользовательских взаимодействий:

* просмотры (`view`)
* добавления в корзину (`addtocart`)
* покупки (`transaction`)

---

## 🎯 Цель проекта

Разработать рекомендательную систему, которая:

* предсказывает товары, интересные пользователю
* учитывает историю взаимодействий
* работает в условиях разреженных данных
* обрабатывает cold start
* может быть развернута как веб-сервис

---

## 🧠 Постановка задачи

Задача формулируется как:

> **ranking / recommendation задача с implicit feedback**

Особенности:

* неявная обратная связь (implicit)
* дисбаланс событий (`view >> addtocart >> transaction`)
* временная структура данных
* высокая разреженность user-item матрицы

---

## 📊 Данные

Датасет состоит из трёх источников:

### 1. `events.csv`

Лог пользовательских событий:

* `timestamp` — время события
* `visitorid` — пользователь
* `event` — тип события (`view`, `addtocart`, `transaction`)
* `itemid` — товар
* `transactionid` — id покупки

---

### 2. `category_tree.csv`

* `categoryid` — категория
* `parentid` — родительская категория

---

### 3. `item_properties`

Файл разбит на две части:

* `item_properties_part1.csv`
* `item_properties_part2.csv`

Содержит:

* `timestamp`
* `itemid`
* `property`
* `value`

---

📎 **Ссылка на датасет:**
*(добавь сюда ссылку)*

---

## 🏗️ Структура проекта

```bash
ecommerce-recsys/
├── airflow/
│   └── dags/
├── mlflow/
│   ├── mlruns/
│   └── mlflow.db
├── artifacts/
│   └── models/
├── data/
│   ├── raw/
│   └── processed/
├── notebooks/
│   ├── 01_eda.ipynb
│   └── 02_modeling.ipynb
├── src/
│   ├── api/
│   ├── data/
│   ├── models/
│   ├── monitoring/
│   └── utils/
├── scripts/
├── docker/
├── README.md
└── requirements.txt
```

---

## 🔍 Этапы проекта

### 1. EDA

* анализ структуры данных
* распределение событий
* анализ активности пользователей
* популярность товаров
* анализ разреженности
* cold start

📄 `notebooks/01_eda.ipynb`

---

### 2. Моделирование

* baseline: Top Popular
* ALS (collaborative filtering)
* генерация кандидатов
* оценка метрик

📄 `notebooks/02_modeling.ipynb`

---

### 3. Инфраструктура

* MLflow для экспериментов
* Airflow для переобучения
* Docker для деплоя

---

### 4. API

* сервис рекомендаций на FastAPI
* endpoint `/recommend`

---

### 5. Мониторинг

* технические метрики
* метрики качества рекомендаций

---

## 📏 Метрики

Основные:

* Recall@K
* MAP@K
* NDCG@K

---

## ⚙️ Используемые технологии

* Python (pandas, numpy)
* implicit (ALS)
* scikit-learn
* LightGBM (расширение)
* MLflow
* FastAPI
* Docker
* Apache Airflow

---

## ⚠️ Особенности задачи

* сильный дисбаланс событий
* long-tail распределение товаров
* высокая разреженность данных
* cold start пользователей
* необходимость гибридного подхода

---

## 🚀 Запуск проекта

### 1. Установка

```bash
git clone <repo>
cd ecommerce-recsys

python -m venv .venv
source .venv/bin/activate

pip install -r requirements.txt
```

---

### 2. Подготовка данных

Положить данные в:

```bash
data/raw/
```

---

### 3. Запуск EDA

```bash
jupyter notebook
```

---

### 4. MLflow

```bash
bash scripts/run_mlflow.sh
```

---

### 5. API

```bash
uvicorn src.api.app:app --reload
```

---

## 📦 Результат

В результате проекта реализована система, которая:

* учитывает поведение пользователей
* предсказывает интересующие товары
* комбинирует несколько подходов (ALS + популярность)
* готова к деплою и масштабированию


# 1. Импорты

In [1]:
# Стандартная библиотека
import gc
import logging
import os
import pickle
import sys
import warnings
from collections import defaultdict
from pathlib import Path
from typing import List, Tuple
import os
import sys
import warnings

import pandas as pd

# Работа с данными
import numpy as np
import pandas as pd

# Визуализация
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
from plotly.subplots import make_subplots

# Статистика
from scipy.sparse import csr_matrix
from scipy.stats import gaussian_kde

# ML
import lightgbm as lgb
from implicit.als import AlternatingLeastSquares
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

# Файлы и хранилища
import boto3
import pyarrow as pa
import pyarrow.parquet as pq
import s3fs
from dotenv import load_dotenv

# Прочее
import phik
from phik import report, resources
from tqdm.auto import tqdm

In [2]:
# настройки отображения
warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)
pd.options.display.float_format = "{:,.0f}".format

# настройки графиков
%matplotlib inline
%config InlineBackend.figure_format = "retina"

# корень проекта
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

print("PROJECT_ROOT:", os.path.basename(PROJECT_ROOT))
print("src exists:", os.path.isdir(os.path.join(PROJECT_ROOT, "src")))

def to_relative(path, base):
    try:
        return os.path.relpath(path, base)
    except ValueError:
        return path

from src.utils.config import (
    DATA_DIR,
    RAW_DIR,
    PROCESSED_DIR,
    EVENTS_PATH,
    CATEGORY_TREE_PATH,
    ITEM_PROPERTIES_PATH,
    ARTIFACTS_DIR,
    MODELS_DIR,
    MLFLOW_BASE_DIR,
    MLFLOW_DIR,
    MLFLOW_DB_PATH,
    AIRFLOW_DIR,
    AIRFLOW_DAGS_DIR,
)

# S3
S3_BASE = "s3://s3-student-mle-20250927-31ecef0a74/recsys"
S3_DATA_DIR = f"{S3_BASE}/data"
S3_REC_DIR = f"{S3_BASE}/recommendations"

# проверка путей
paths_to_check = {
    "DATA_DIR": DATA_DIR,
    "RAW_DIR": RAW_DIR,
    "PROCESSED_DIR": PROCESSED_DIR,
    "ARTIFACTS_DIR": ARTIFACTS_DIR,
    "MODELS_DIR": MODELS_DIR,
    "MLFLOW_BASE_DIR": MLFLOW_BASE_DIR,
    "MLFLOW_DIR": MLFLOW_DIR,
    "MLFLOW_DB_PATH": MLFLOW_DB_PATH,
    "AIRFLOW_DIR": AIRFLOW_DIR,
    "AIRFLOW_DAGS_DIR": AIRFLOW_DAGS_DIR,
    "EVENTS_PATH": EVENTS_PATH,
    "CATEGORY_TREE_PATH": CATEGORY_TREE_PATH,
    "ITEM_PROPERTIES_PATH": ITEM_PROPERTIES_PATH,
}

print("\nProject paths:\n")

for name, path in paths_to_check.items():
    rel_path = to_relative(path, PROJECT_ROOT)
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"{name:<22} {rel_path:<40} [{status}]")

PROJECT_ROOT: ecommerce-recsys
src exists: True

Project paths:

DATA_DIR               data                                     [OK]
RAW_DIR                data/raw                                 [OK]
PROCESSED_DIR          data/processed                           [OK]
ARTIFACTS_DIR          artifacts                                [OK]
MODELS_DIR             artifacts/models                         [OK]
MLFLOW_BASE_DIR        mlflow                                   [OK]
MLFLOW_DIR             mlflow/mlruns                            [OK]
MLFLOW_DB_PATH         mlflow/mlflow.db                         [MISSING]
AIRFLOW_DIR            airflow                                  [OK]
AIRFLOW_DAGS_DIR       airflow/dags                             [OK]
EVENTS_PATH            data/raw/events.csv                      [OK]
CATEGORY_TREE_PATH     data/raw/category_tree.csv               [OK]
ITEM_PROPERTIES_PATH   data/raw/item_properties.csv             [MISSING]


# 2. Загрузка данных

In [19]:
events = pd.read_csv(
    EVENTS_PATH,
    dtype={
        "visitorid": "int32",
        "event": "category",
        "itemid": "int32",
        "transactionid": "float64",
    },
)

category_tree = pd.read_csv(
    f"{RAW_DIR}/category_tree.csv",
    dtype={
        "categoryid": "int32",
        "parentid": "float64",
    },
)

item_props_part1 = pd.read_csv(
    f"{RAW_DIR}/item_properties_part1.csv",
    dtype={
        "itemid": "int32",
        "property": "category",
        "value": "string",
    },
)

item_props_part2 = pd.read_csv(
    f"{RAW_DIR}/item_properties_part2.csv",
    dtype={
        "itemid": "int32",
        "property": "category",
        "value": "string",
    },
)

item_props = pd.concat([item_props_part1, item_props_part2], ignore_index=True)

print("Events shape:", events.shape)
print("Category tree shape:", category_tree.shape)
print("Item properties shape:", item_props.shape)

Events shape: (2756101, 5)
Category tree shape: (1669, 2)
Item properties shape: (20275902, 4)


# 3. Обзор данных

Проверяем данные, есть ли с ними явные проблемы.

## events

In [ ]:
# events - обзор датасета
events.head()

,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,NaN
1,1433224214164,992329,view,248676,NaN
2,1433221999827,111016,view,318965,NaN
3,1433221955914,483717,view,253185,NaN
4,1433221337106,951259,view,367447,NaN


In [ ]:
# events - описание
events.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype   
---  ------         -----   
 0   timestamp      int64   
 1   visitorid      int32   
 2   event          category
 3   itemid         int32   
 4   transactionid  float64 
dtypes: category(1), float64(1), int32(2), int64(1)
memory usage: 65.7 MB


In [ ]:
# events - числовые признаки, основные статистики
events.describe()

,timestamp,visitorid,itemid,transactionid
count,"2,756,101","2,756,101","2,756,101","22,457"
mean,"1,436,424,488,348","701,923","234,922","8,826"
std,"3,366,312,180","405,688","134,195","5,099"
min,"1,430,622,004,384",0,3,0
25%,"1,433,478,194,792","350,566","118,120","4,411"
50%,"1,436,453,013,599","702,060","236,067","8,813"
75%,"1,439,225,105,168","1,053,437","350,715","13,224"
max,"1,442,545,187,788","1,407,579","466,867","17,671"


In [46]:
# events - категориальные признаки, основные статистики
display(events.describe(include="category"))
display(events["event"].value_counts())

,event
count,2756101
unique,3
top,view
freq,2664312


event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64

### Выводы по events

Датасет представляет собой лог пользовательских событий (implicit feedback) с выраженным дисбалансом типов взаимодействий. Основную долю составляют просмотры (~97%), тогда как добавления в корзину (~2.5%) и покупки (~0.8%) являются редкими событиями.

Это определяет подход к решению задачи:
- нельзя использовать только покупки из-за их редкости;
- целевым действием выбрано addtocart как баланс между частотой и значимостью;
- требуется учитывать разные типы взаимодействий с разными весами.

Данные имеют временную структуру, что позволяет использовать временные признаки и требует time-based разбиения на train/validation/test.

Таким образом, датасет подходит для построения двухстадийной рекомендательной системы:
- генерация кандидатов (ALS, popularity, co-visitation);
- ранжирование (градиентный бустинг).

## category_tree

In [ ]:
# category_tree - обзор датасета
category_tree.head()

,categoryid,parentid
0,1016,213
1,809,169
2,570,9
3,1691,885
4,536,"1,691"


In [ ]:
# category_tree - описание
category_tree.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1669 entries, 0 to 1668
Data columns (total 2 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   categoryid  1669 non-null   int32  
 1   parentid    1644 non-null   float64
dtypes: float64(1), int32(1)
memory usage: 19.7 KB


In [ ]:
# category_tree - основные статистики
category_tree.describe()

,categoryid,parentid
count,"1,669","1,644"
mean,849,848
std,490,505
min,0,8
25%,427,381
50%,848,866
75%,"1,273","1,291"
max,"1,698","1,698"


### Выводы по category_tree

Датасет category_tree представляет собой иерархическую структуру категорий в виде дерева, где каждая категория связана с родительской через поле parentid.

Важно отметить, что данный датасет не содержит категориальных признаков в классическом понимании (как признаки модели), а является справочником (metadata), описывающим структуру категорий.

Для использования в модели необходимо:
- связать товары с категориями через item_properties;
- затем использовать categoryid и производные признаки (например, родительские категории) как признаки.

Наличие иерархии позволяет строить дополнительные признаки:
- уровень категории;
- родительская категория;
- близость товаров по дереву категорий.

Таким образом, category_tree служит вспомогательной таблицей для feature engineering, а не источником готовых признаков.

## item_props

In [53]:
# item_props - обзор датасета
item_props.head()

,timestamp,itemid,property,value
0,1435460400000,460429,categoryid,1338
1,1441508400000,206783,888,1116713 960601 n277.200
2,1439089200000,395014,400,n552.000 639502 n720.000 424566
3,1431226800000,59481,790,n15360.000
4,1431831600000,156781,917,828513


In [54]:
# item_props - описание
item_props.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20275902 entries, 0 to 20275901
Data columns (total 4 columns):
 #   Column     Dtype 
---  ------     ----- 
 0   timestamp  int64 
 1   itemid     int32 
 2   property   object
 3   value      string
dtypes: int32(1), int64(1), object(1), string(1)
memory usage: 541.4+ MB


In [55]:
# item_props - основные статистики
item_props.describe()

,timestamp,itemid
count,"20,275,902","20,275,902"
mean,"1,435,156,943,683","233,390"
std,"3,327,797,781","134,845"
min,"1,431,226,800,000",0
25%,"1,432,436,400,000","116,516"
50%,"1,433,646,000,000","233,483"
75%,"1,437,879,600,000","350,304"
max,"1,442,113,200,000","466,866"


In [61]:
# events - категориальные признаки, основные статистики
display(item_props.describe(include="object"))
display(item_props.describe(include="string"))
display(item_props["property"].value_counts())
display(item_props["value"].value_counts())

,property
count,20275902
unique,1104
top,888
freq,3000398


,value
count,20275902
unique,1966868
top,769062
freq,1537247


property
888           3000398
790           1790516
available     1503639
categoryid     788214
6              631471
               ...   
782                 1
288                 1
722                 1
744                 1
769                 1
Name: count, Length: 1104, dtype: int64

value
769062          1537247
0                863246
1                641420
679677           417054
1285872          417053
                 ...   
127255                1
90224                 1
860500                1
558168                1
n1020780.000          1
Name: count, Length: 1966868, dtype: Int64

### Выводы по item_properties

Датасет item_properties представляет собой набор свойств товаров в формате key-value (Entity–Attribute–Value), содержащий более 20 млн записей. Каждый товар может иметь множество свойств, заданных через пары (property, value).

Данные характеризуются:
- высокой размерностью (более 1100 различных свойств);
- высокой кардинальностью значений (почти 2 млн уникальных значений);
- отсутствием единого формата в колонке value (числа, строки, смешанные значения);
- наличием временной динамики свойств.

В текущем виде датасет не может быть напрямую использован в модели и требует значительной предобработки:
- фильтрации и отбора наиболее информативных свойств;
- приведения значений к числовому или категориальному виду;
- агрегации до уровня товара (item-level features).

Особое значение имеют свойства:
- categoryid — для связывания с категорией товара;
- available — потенциально бинарный признак доступности.

Таким образом, item_properties является ключевым источником признаков товаров, но требует сложного feature engineering.

# 4. Типы данных, пропуски, дубликаты

In [63]:
def analyze_dataframe(df, subset=None):
    """
    Выполняет сводный анализ одного DataFrame.

    Параметры:
    df : pd.DataFrame
        Входной датафрейм для анализа.
    subset : list[str] | None, default=None
        Список колонок для проверки дубликатов по подмножеству признаков.
        Если None, дополнительно считаются только полные дубликаты строк.

    Возвращает:
    tuple[pd.DataFrame, dict]
        summary : pd.DataFrame
            Таблица со статистикой по каждому столбцу:
            - dtype: тип данных
            - missing_count: количество пропусков
            - missing_percent: доля пропусков, %
            - non_null_count: количество непустых значений
            - memory_MB: объем памяти столбца в MB
            - n_unique: число уникальных значений

        duplicates_info : dict
            Словарь с общей информацией по датафрейму:
            - row_count: число строк
            - column_count: число столбцов
            - total_memory_MB: общий объем памяти, MB
            - full_duplicates_count: число полных дубликатов
            - full_duplicates_percent: доля полных дубликатов, %
            - subset_duplicates_count: число дубликатов по subset, если subset передан
            - subset_duplicates_percent: доля дубликатов по subset, если subset передан

    Примечания:
    - Для object-столбцов функция пытается корректно считать уникальные значения,
      включая случай, когда в ячейках находятся списки или numpy-массивы.
    - Если датафрейм пустой, доли пропусков и дубликатов возвращаются как 0.
    """
    row_count = len(df)
    column_count = df.shape[1]
    total_memory_mb = df.memory_usage(deep=True).sum() / (1024**2)

    summary = pd.DataFrame(
        {
            "dtype": df.dtypes.astype(str),
            "missing_count": df.isna().sum(),
            "missing_percent": (
                (df.isna().sum() / row_count * 100) if row_count > 0 else 0
            ),
            "non_null_count": df.notna().sum(),
            "memory_MB": df.memory_usage(deep=True, index=False) / (1024**2),
        }
    )

    unique_values = {}

    for col in df.columns:
        series = df[col]

        if series.dtype == "object":
            non_null = series.dropna()

            if not non_null.empty and isinstance(non_null.iloc[0], (list, np.ndarray)):
                unique_values[col] = series.explode().dropna().nunique()
            else:
                unique_values[col] = series.nunique()
        else:
            unique_values[col] = series.nunique()

    summary["n_unique"] = pd.Series(unique_values)

    summary["missing_count"] = summary["missing_count"].astype("int64")
    summary["non_null_count"] = summary["non_null_count"].astype("int64")
    summary["n_unique"] = summary["n_unique"].astype("int64")
    summary["missing_percent"] = summary["missing_percent"].round(4)
    summary["memory_MB"] = summary["memory_MB"].round(4)

    full_duplicates_count = int(df.duplicated().sum())
    full_duplicates_percent = (
        round(full_duplicates_count / row_count * 100, 4) if row_count > 0 else 0.0
    )

    duplicates_info = {
        "row_count": row_count,
        "column_count": column_count,
        "total_memory_MB": round(total_memory_mb, 4),
        "full_duplicates_count": full_duplicates_count,
        "full_duplicates_percent": full_duplicates_percent,
    }

    if subset is not None:
        subset_duplicates_count = int(df.duplicated(subset=subset).sum())
        subset_duplicates_percent = (
            round(subset_duplicates_count / row_count * 100, 4)
            if row_count > 0
            else 0.0
        )

        duplicates_info["subset"] = subset
        duplicates_info["subset_duplicates_count"] = subset_duplicates_count
        duplicates_info["subset_duplicates_percent"] = subset_duplicates_percent

    return summary.sort_values("missing_count", ascending=False), duplicates_info


def build_dataset_summary(datasets):
    """
    Формирует сводную таблицу по нескольким датафреймам.

    Параметры:
    datasets : dict
        Словарь формата:
        {
            "dataset_name": {
                "df": pd.DataFrame,
                "subset": list[str] | None
            }
        }

        Пример:
        {
            "events": {
                "df": events,
                "subset": ["timestamp", "visitorid", "event", "itemid", "transactionid"]
            },
            "category_tree": {
                "df": category_tree,
                "subset": ["categoryid"]
            }
        }

    Возвращает:
    pd.DataFrame
        Сводная таблица уровня датасета со столбцами:
        - dataset
        - row_count
        - column_count
        - total_memory_MB
        - total_missing_count
        - total_missing_percent
        - full_duplicates_count
        - full_duplicates_percent
        - subset_duplicates_count
        - subset_duplicates_percent
        - n_object_cols
        - n_category_cols
        - n_numeric_cols
    """
    rows = []

    for name, config in datasets.items():
        df = config["df"]
        subset = config.get("subset")

        summary, duplicates_info = analyze_dataframe(df, subset=subset)

        row_count = len(df)
        total_cells = df.shape[0] * df.shape[1]
        total_missing_count = int(df.isna().sum().sum())
        total_missing_percent = (
            round(total_missing_count / total_cells * 100, 4)
            if total_cells > 0
            else 0.0
        )

        row = {
            "dataset": name,
            "row_count": duplicates_info["row_count"],
            "column_count": duplicates_info["column_count"],
            "total_memory_MB": duplicates_info["total_memory_MB"],
            "total_missing_count": total_missing_count,
            "total_missing_percent": total_missing_percent,
            "full_duplicates_count": duplicates_info["full_duplicates_count"],
            "full_duplicates_percent": duplicates_info["full_duplicates_percent"],
            "subset_duplicates_count": duplicates_info.get(
                "subset_duplicates_count", np.nan
            ),
            "subset_duplicates_percent": duplicates_info.get(
                "subset_duplicates_percent", np.nan
            ),
            "n_object_cols": int((df.dtypes == "object").sum()),
            "n_category_cols": int((df.dtypes.astype(str) == "category").sum()),
            "n_numeric_cols": int(df.select_dtypes(include=[np.number]).shape[1]),
        }

        rows.append(row)

    return (
        pd.DataFrame(rows)
        .sort_values("total_memory_MB", ascending=False)
        .reset_index(drop=True)
    )

In [65]:
datasets = {
    "events": {
        "df": events,
        "subset": ["timestamp", "visitorid", "event", "itemid", "transactionid"],
    },
    "category_tree": {"df": category_tree, "subset": ["categoryid"]},
    "item_props": {
        "df": item_props,
        "subset": ["timestamp", "itemid", "property", "value"],
    },
}

dataset_summary = build_dataset_summary(datasets)
display(dataset_summary.T)

,0,1,2
dataset,item_props,events,category_tree
row_count,20275902,2756101,1669
column_count,4,5,2
total_memory_MB,"2,539",66,0
total_missing_count,0,2733644,25
total_missing_percent,0,20,1
full_duplicates_count,0,460,0
full_duplicates_percent,0,0,0
subset_duplicates_count,0,460,0
subset_duplicates_percent,0,0,0


### Общие выводы по данным

Данные состоят из трёх датасетов, существенно различающихся по размеру и роли в задаче.

Наиболее крупным и сложным является датасет item_properties (~20 млн строк, ~2.5 GB), который содержит признаки товаров в формате key-value. Несмотря на отсутствие пропусков, он требует значительной предобработки из-за высокой кардинальности и неструктурированного формата значений.

Датасет events (~2.7 млн строк) содержит пользовательские взаимодействия и является основным источником сигналов для обучения модели. Наличие пропусков связано с особенностями данных (редкие покупки) и не является проблемой.

Датасет category_tree представляет собой компактный справочник категорий и используется для построения дополнительных признаков через иерархию.

Анализ дубликатов показал их незначительное количество, что не оказывает существенного влияния на модель.

Таким образом, основная сложность проекта заключается не в качестве данных (пропусках или дубликатах), а в их структуре и масштабе, что требует построения пайплайна feature engineering и использования двухстадийной архитектуры рекомендательной системы.

# 6. Проверка связности таблиц

# 5. Уникальные значения

In [66]:
print("Users:", events["visitorid"].nunique())
print("Items:", events["itemid"].nunique())
print("Events:", events.shape[0])

Users: 1407580
Items: 235061
Events: 2756101


# 6. Распределение типов событий

In [ ]:
events["event"].value_counts()

In [ ]:
sns.countplot(data=events, x="event")
plt.title("Event distribution")
plt.show()

In [ ]:
Вывод:
обычно view >> addtocart >> transaction
сильный дисбаланс

### Общие выводы

1. Данные представляют классическую задачу рекомендательной системы:
   - implicit feedback
   - высокая разреженность
   - сильный дисбаланс событий

2. Основной таргет:
   - add-to-cart (addtocart)

3. Архитектура решения:
   - двухэтапная модель:
     - candidate generation (ALS, popular, co-visitation)
     - ranking (LightGBM)

4. Основные источники признаков:
   - события пользователей (events)
   - категории товаров (category_tree)
   - свойства товаров (item_properties)

5. Ключевые сложности:
   - cold-start пользователи
   - long-tail распределение товаров
   - обработка item_properties

6. Обоснование методов:
   - ALS — для работы с разреженными матрицами
   - Popular — сильный baseline
   - Ranker — для финальной персонализации


# 7. Временной анализ

In [ ]:
events["datetime"] = pd.to_datetime(events["timestamp"], unit="ms")

events["date"] = events["datetime"].dt.date
events["hour"] = events["datetime"].dt.hour

## События по времени

In [ ]:
events.groupby("date").size().plot(figsize=(12,4), title="Events over time")
plt.show()

## По часам

In [ ]:
sns.countplot(data=events, x="hour")
plt.title("Events by hour")
plt.show()

Вывод:
пики активности
есть ли сезонность

# 8. Популярные товары

In [ ]:
top_items = events["itemid"].value_counts().head(20)

top_items.plot(kind="bar", figsize=(10,4), title="Top items")
plt.show()

Вывод:
есть сильный popularity bias

# 9. Активность пользователей

In [ ]:
user_activity = events.groupby("visitorid").size()

user_activity.describe()

In [ ]:
sns.histplot(user_activity, bins=50, log_scale=True)
plt.title("User activity distribution")
plt.show()

In [ ]:
Вывод:
большинство пользователей малоактивны
длинный хвост

# 10. Активность товаров

In [ ]:
item_activity = events.groupby("itemid").size()

sns.histplot(item_activity, bins=50, log_scale=True)
plt.title("Item popularity distribution")
plt.show()

In [ ]:
Вывод:
few popular items, many rare

# 11. Конверсия

In [ ]:
events["event"].value_counts(normalize=True)

In [ ]:
Вывод:
низкая конверсия в покупку
add-to-cart — разумный target

# 12. Разреженность (sparsity)

In [ ]:
n_users = events["visitorid"].nunique()
n_items = events["itemid"].nunique()
n_interactions = len(events)

sparsity = 1 - n_interactions / (n_users * n_items)

print("Sparsity:", sparsity)

In [ ]:
Вывод:
обычно ~99%+
значит нужен CF/ALS

# 13. Cold start

In [ ]:
user_counts = events["visitorid"].value_counts()
cold_users = (user_counts == 1).mean()

print("Cold users ratio:", cold_users)

In [ ]:
Вывод:
много новых пользователей
нужен popular fallback

# 14. Анализ item_properties

In [ ]:
item_props["property"].value_counts().head(10)

In [ ]:
item_props["value"].nunique()

In [ ]:
Вывод:
данные разреженные
можно использовать как фичи

# Финальные выводы

In [ ]:
1. Данные имеют сильный дисбаланс: большинство событий — просмотры.
2. Пользовательская активность распределена неравномерно (long tail).
3. Наблюдается сильный popularity bias среди товаров.
4. Матрица user-item сильно разрежена (>99%).
5. Высокая доля cold users → нужен fallback (Top Popular).
6. Добавление в корзину выбрано как целевое действие.
7. Подход к решению:
   - candidate generation (ALS, popularity)
   - ranking (LightGBM)